# Experiment 02 — Language Comprehension

```
Language comprehension
      ↓
Meaning + memory + emotion + social context
      ↓
Response planning
      ↓
Word and sentence construction
      ↓
Motor commands
```

This is the first stage of a 5-stage toy pipeline sketching (very loosely) how a
human turns heard/read language into a spoken response. Each stage gets its own
notebook, its own small neural network, and its own toy task — **there's no real
wiring between them yet** (this stage's output is a vector shape, not literally fed
into stage 03's notebook). That's intentional: the point right now is "does a small
network learn a plausible piece of the job," not an end-to-end system. Wiring them
together for real is a later step.

**This stage's job:** take a sentence and produce two things — an **intent** (what
kind of utterance is this: a question, a statement, a greeting, a command?) and a
**comprehension vector** (a fixed-size numeric summary of the sentence) that a later
stage could consume. We'll train a small network on a tiny hand-labeled toy set and
be upfront about what it did and didn't learn from just 16 examples.

In [1]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

## A tiny labeled toy set

16 short sentences, 4 per intent. Small enough to read in one glance, small enough
that we should be honest later about what generalizes and what doesn't.

In [2]:
sentences_by_intent = {
    "question": [
        "what time is it",
        "where are you going",
        "how are you today",
        "why is the sky blue",
    ],
    "statement": [
        "i am hungry now",
        "the sky is blue",
        "she walked to school",
        "dogs are loyal animals",
    ],
    "greeting": [
        "hello there friend",
        "good morning everyone",
        "hi how are you",
        "nice to meet you",
    ],
    "command": [
        "please close the door",
        "sit down now please",
        "turn off the lights",
        "bring me some water",
    ],
}
intents = list(sentences_by_intent.keys())

train_data = [(s, intent) for intent, sents in sentences_by_intent.items() for s in sents]
random.shuffle(train_data)

vocab = {"<unk>": 0}
for s, _ in train_data:
    for w in s.split():
        vocab.setdefault(w, len(vocab))

print(f"{len(train_data)} training sentences, {len(intents)} intents, vocab size {len(vocab)}")

16 training sentences, 4 intents, vocab size 47


## Architecture: embed, pool, classify

Each word gets a learned embedding; a sentence's embeddings are mean-pooled into one
vector, then a small feed-forward layer turns that into a **comprehension vector** —
this hidden layer is the thing a downstream stage would actually consume, not the
final intent label. The intent label is just how we force the network to learn
something structured in that hidden layer at all (there's no way to get a good
"comprehension vector" without training it to be useful for *something*).

In [3]:
class LanguageComprehension(nn.Module):
    def __init__(self, vocab_size, n_intents, embed_dim=16, hidden_dim=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.to_comprehension = nn.Linear(embed_dim, hidden_dim)
        self.to_intent = nn.Linear(hidden_dim, n_intents)

    def forward(self, token_ids):
        # token_ids: (batch, seq_len), padded with 0 (<unk>) — fine for mean pooling
        # since real words dominate the average as long as sentences aren't tiny.
        word_vecs = self.embed(token_ids)          # (batch, seq_len, embed_dim)
        pooled = word_vecs.mean(dim=1)              # (batch, embed_dim)
        comprehension = torch.tanh(self.to_comprehension(pooled))  # (batch, hidden_dim)
        logits = self.to_intent(comprehension)       # (batch, n_intents)
        return logits, comprehension


def encode(sentence, vocab):
    return [vocab.get(w, vocab["<unk>"]) for w in sentence.split()]


def to_batch(data, vocab):
    encoded = [encode(s, vocab) for s, _ in data]
    max_len = max(len(e) for e in encoded)
    padded = [e + [vocab["<unk>"]] * (max_len - len(e)) for e in encoded]
    labels = [intents.index(intent) for _, intent in data]
    return torch.tensor(padded), torch.tensor(labels)


model = LanguageComprehension(vocab_size=len(vocab), n_intents=len(intents))
X, y = to_batch(train_data, vocab)
print("batch shape:", X.shape, "labels shape:", y.shape)

batch shape: torch.Size([16, 5]) labels shape: torch.Size([16])


## Training

Straightforward cross-entropy on intent, Adam, a few hundred steps over this tiny fixed batch.

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

losses = []
for step in range(300):
    logits, _ = model(X)
    loss = F.cross_entropy(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

with torch.no_grad():
    logits, _ = model(X)
    train_acc = (logits.argmax(dim=1) == y).float().mean().item()

print(f"loss: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"train accuracy: {train_acc:.2%}")

loss: 1.382 -> 0.000
train accuracy: 100.00%


## Comprehending something new

The real test isn't the training accuracy above (16 examples, 4 classes — memorizing
is easy) — it's what happens on a sentence the network never saw. We'll try two
in-distribution-ish sentences (all words known, new combination) and one with two
genuinely unseen words falling back to `<unk>`.

In [5]:
def comprehend(sentence, model, vocab):
    ids = torch.tensor([encode(sentence, vocab)])
    with torch.no_grad():
        logits, comprehension = model(ids)
        probs = F.softmax(logits, dim=1).squeeze(0)
    predicted = intents[probs.argmax().item()]
    return predicted, probs, comprehension.squeeze(0)


test_sentences = [
    "how are you",              # known words, new combination (seen in "how are you today" / "hi how are you")
    "please bring the lights",  # known words, new combination (mixes "command" vocab)
    "when will you arrive",     # "when" and "arrive" are unseen -> fall back to <unk>
]

for s in test_sentences:
    predicted, probs, vec = comprehend(s, model, vocab)
    top_prob = probs.max().item()
    print(f"{s!r:35} -> {predicted:10} (p={top_prob:.2f})  |comprehension|={vec.norm():.3f}")

'how are you'                       -> question   (p=1.00)  |comprehension|=3.615
'please bring the lights'           -> command    (p=1.00)  |comprehension|=3.769
'when will you arrive'              -> greeting   (p=1.00)  |comprehension|=3.567


## What this hands off (conceptually)

A `comprehension` vector — 16 real numbers per sentence — plus a soft intent
distribution. Stage 03 (`meaning + memory + emotion + social context`) would take a
vector shaped like this as one of *four* inputs it fuses together; for now stage 03
just makes up its own standalone stand-in vectors rather than importing this model,
per the "no real connection yet" scope.

The honest result worth sitting with: `how are you` and `please bring the lights`
both get classified correctly at p=1.00 by recombining known words in new ways —
genuine, if modest, generalization. But `when will you arrive`, where "when" and
"arrive" both fall back to `<unk>`, gets classified as **greeting** with the *same*
p=1.00 confidence — confidently wrong, not uncertain. With only 16 training
sentences the network never learned to be unsure about words it doesn't know; it
just treats `<unk>` as another token with a definite meaning. A real system needs
either a much bigger vocabulary or an explicit way to signal low confidence on
unknown words — this toy doesn't have either.